# Avaliacao completa das pixel heads

Teste completo do INSPLAD-seg, sem limite de imagens e sem descartar bordas. Avaliacao **descritiva**, nao um holdout independente: parte destas imagens ja foi usada para escolher epocas e receitas.

Selecione o kernel Conda `anomaly_attentideffernet`. Edite os caminhos na proxima celula e execute em ordem. Receitas e estatisticas sao lidas dos checkpoints; ausencias nao sao preenchidas silenciosamente com defaults antigos.

**Precisao e custo:** `histogram` percorre todos os pixels, mas aproxima AUROC/AP e os limiares por bins. `exact` usa sklearn sobre todos os pixels e pode exigir muitos GB de RAM. AUPRO global e calculado separadamente com a implementacao CFLOW-AD; zero nao e corrigido automaticamente. Melhores F1 medidos no teste sao oraculos diagnosticos, nao limiares de producao. Scores NF nao sao probabilidades calibradas.

In [12]:
from pathlib import Path
import sys
from datetime import datetime

candidates = [Path.cwd(), Path.cwd() / 'anomaly-detection-pesquisa' / 'differnet', *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'core' / 'cflow.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Abra o notebook a partir do projeto differnet ou informe PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))
DATASET_ROOT = Path(r'C:\Users\teo-s\Documents\GitHub\anomaly-detection-dataset\insplad-seg-no-back-treino')
NF_ROOT = PROJECT_ROOT / 'resultado_analise_final' / 'treino_nf_heads' / 'run_20261026_102309'
SE_ROOT = PROJECT_ROOT / 'final_models' / 'SEDiffernet_noBg'
CLASSES = ['glass-insulator', 'lightning-rod-suspension', 'polymer-insulator-upper-shackle', 'vari-grip', 'yoke-suspension']
FALLBACK_RECIPE = None  # None exige metadados; alternativa explicita: 'general', 'best_per_class', 'baseline'.
PIXEL_BACKEND = 'histogram'  # 'exact' pode ultrapassar a RAM disponivel.
N_BINS = 65536
SAMPLE_PER_IMAGE = 1024
BOOTSTRAP = 200
SEED = 42
EXAMPLES_PER_CLASS = 6
COMPUTE_AUPRO = True
COMPUTE_LEGACY_AUPRO = False
KEEP_MAPS = False
OUTPUT_DIR = PROJECT_ROOT / 'resultado_analise_final' / 'pixel_notebook' / datetime.now().strftime('run_%Y%m%d_%H%M%S')
assert PIXEL_BACKEND in ('histogram', 'exact')
assert N_BINS >= 32 and SAMPLE_PER_IMAGE > 0 and BOOTSTRAP >= 0
print('Projeto:', PROJECT_ROOT)
print('Heads:', NF_ROOT)
print('Saida:', OUTPUT_DIR)
print('Backend de pixels:', PIXEL_BACKEND, '| teste completo, stride=1, margem=0')

Projeto: c:\Users\teo-s\Documents\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet
Heads: c:\Users\teo-s\Documents\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\resultado_analise_final\treino_nf_heads\run_20261026_102309
Saida: c:\Users\teo-s\Documents\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\resultado_analise_final\pixel_notebook\run_20261026_180544
Backend de pixels: histogram | teste completo, stride=1, margem=0


In [13]:
import gc
import hashlib
import json
import math
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from IPython.display import display, Markdown
from tqdm.auto import tqdm
import config as c
from core.cflow import (CFlowPixelHead, SEBackboneFeatureExtractor, read_cflow_hparams,
                        infer_n_blocks, nll_with_tta, compute_positional_stats,
                        gaussian_smooth, image_score_from_map)
from core.eval_pipeline import (build_eval_datasets, eval_loader, load_se_model,
                                find_nf_head_checkpoint, compute_aupro_cflow,
                                compute_pro_rd_style, eval_transform)
from scripts.train.train_all_nf_heads import resolve_se_checkpoints
from scripts.train.pixel_train_from_pretrained import resolve_training_recipe

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python:', sys.executable)
print('PyTorch:', torch.__version__, '| dispositivo:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
pd.set_option('display.max_columns', 40)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_path(path):
    return str(path).replace('\\', '/').lower()


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else float('nan')


def confusion_metrics(tp, fp, fn, tn):
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    f1 = safe_ratio(2 * tp, 2 * tp + fp + fn)
    normal_f1 = safe_ratio(2 * tn, 2 * tn + fp + fn)
    iou = safe_ratio(tp, tp + fp + fn)
    normal_iou = safe_ratio(tn, tn + fp + fn)
    total = tp + fp + fn + tn
    weighted_f1 = safe_ratio(((tp + fn) * f1 if tp + fn else 0.) +
                             ((tn + fp) * normal_f1 if tn + fp else 0.), total)
    return dict(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                accuracy=safe_ratio(tp + tn, total), precision=precision, recall=recall,
                specificity=specificity, balanced_accuracy=(recall + specificity) / 2,
                f1=f1, dice=f1, f1_normal=normal_f1,
                f1_macro_binary=float(np.nanmean([f1, normal_f1])), f1_weighted=weighted_f1,
                f1_micro_binary=safe_ratio(tp + tn, total),
                iou=iou, iou_normal=normal_iou, miou=float(np.nanmean([iou, normal_iou])),
                fpr=safe_ratio(fp, fp + tn))


def curve_from_counts(positive, negative):
    tp = np.r_[0., np.cumsum(positive[::-1], dtype=np.float64)]
    fp = np.r_[0., np.cumsum(negative[::-1], dtype=np.float64)]
    recall = tp / positive.sum() if positive.sum() else np.full(tp.shape, np.nan)
    fpr = fp / negative.sum() if negative.sum() else np.full(fp.shape, np.nan)
    precision = np.divide(tp, tp + fp, out=np.ones(tp.shape), where=(tp + fp) > 0)
    f1 = np.divide(2 * tp, positive.sum() + tp + fp,
                   out=np.zeros(tp.shape), where=(positive.sum() + tp + fp) > 0)
    return dict(fpr=fpr, recall=recall, precision=precision, f1=f1,
                auroc=float(np.trapz(recall, fpr)) if positive.sum() and negative.sum() else np.nan,
                ap=float(np.sum(np.diff(recall) * precision[1:])) if positive.sum() else np.nan)


def binary_image_metrics(labels, scores, threshold):
    labels, scores = np.asarray(labels), np.asarray(scores)
    predicted = scores >= threshold
    result = confusion_metrics(np.sum(predicted & (labels == 1)), np.sum(predicted & (labels == 0)),
                               np.sum(~predicted & (labels == 1)), np.sum(~predicted & (labels == 0)))
    both = len(np.unique(labels)) == 2
    result.update(auroc=roc_auc_score(labels, scores) if both else np.nan,
                  ap=average_precision_score(labels, scores) if both else np.nan)
    return result


def fit_image_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds) or len(np.unique(labels)) < 2:
        return float(np.median(scores))
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.nanargmax(f1)])

Python: c:\Users\teo-s\.conda\envs\anomaly_attentideffernet\python.exe
PyTorch: 2.5.1+cu121 | dispositivo: cuda
NVIDIA GeForce RTX 4060 Laptop GPU


In [14]:
def pixel_histograms(maps, masks, validation_indices):
    rng = np.random.default_rng(SEED)
    samples = [np.asarray(score).ravel()[rng.integers(0, score.size, SAMPLE_PER_IMAGE)] for score in maps]
    sample = np.concatenate(samples)
    edges = np.unique(np.quantile(sample, np.linspace(0, 1, N_BINS + 1)[1:-1]))
    slots = len(edges) + 1
    positive, negative = np.zeros(slots), np.zeros(slots)
    val_positive, val_negative = np.zeros(slots), np.zeros(slots)
    group_size = max(1, math.ceil(slots / 2048))
    starts = np.arange(0, slots, group_size)
    boot_positive = np.zeros((len(maps), len(starts)), dtype=np.uint32)
    boot_negative = np.zeros_like(boot_positive)
    per_image = []
    for index, (score, mask) in enumerate(tqdm(zip(maps, masks), total=len(maps), desc='Histogramas completos')):
        labels = np.asarray(mask).ravel().astype(bool)
        bins = np.searchsorted(edges, np.asarray(score).ravel(), side='right')
        pos = np.bincount(bins[labels], minlength=slots)
        neg = np.bincount(bins[~labels], minlength=slots)
        positive += pos
        negative += neg
        if index in validation_indices:
            val_positive += pos
            val_negative += neg
        boot_positive[index] = np.add.reduceat(pos, starts)
        boot_negative[index] = np.add.reduceat(neg, starts)
        per_image.append(curve_from_counts(pos, neg)['auroc'])
    curve = curve_from_counts(positive, negative)
    thresholds = np.r_[np.inf, edges[::-1], -np.inf]
    val_curve = curve_from_counts(val_positive, val_negative)
    source = 'validation_reused_in_full_test'
    if not val_positive.sum() or not val_negative.sum():
        val_curve, source = curve, 'test_oracle_no_validation_metadata'
    selected = int(np.nanargmax(val_curve['f1']))
    threshold = float(thresholds[selected])
    oracle_index = int(np.nanargmax(curve['f1']))
    result = dict(pixel_auroc=curve['auroc'], pixel_ap=curve['ap'],
                  pixel_prevalence=safe_ratio(positive.sum(), positive.sum() + negative.sum()),
                  pixel_best_f1_test_oracle=float(curve['f1'][oracle_index]),
                  pixel_threshold_test_oracle=float(thresholds[oracle_index]),
                  pixel_threshold=threshold, threshold_source=source,
                  pixel_ranking_backend=PIXEL_BACKEND, pixel_bins=slots,
                  pixel_threshold_backend='histogram')
    if PIXEL_BACKEND == 'exact':
        labels = np.asarray(masks).reshape(-1)
        scores = np.asarray(maps).reshape(-1)
        if labels.min() != labels.max():
            result['pixel_auroc'] = float(roc_auc_score(labels, scores))
            result['pixel_ap'] = float(average_precision_score(labels, scores))
    return result, curve, thresholds, boot_positive, boot_negative, per_image, sample


def image_bootstrap(labels, scores, positive, negative):
    if not BOOTSTRAP:
        return {}
    rng = np.random.default_rng(SEED)
    groups = [np.flatnonzero(labels == label) for label in (0, 1)]
    values = []
    for _ in tqdm(range(BOOTSTRAP), desc='Bootstrap por imagem'):
        indices = np.concatenate([rng.choice(group, len(group), replace=True) for group in groups if len(group)])
        curve = curve_from_counts(positive[indices].sum(axis=0), negative[indices].sum(axis=0))
        values.append([curve['auroc'], curve['ap'],
                       roc_auc_score(labels[indices], scores[indices]) if len(np.unique(labels[indices])) == 2 else np.nan])
    limits = np.nanpercentile(values, [2.5, 97.5], axis=0)
    return {f'{name}_ci_{bound}': float(limits[side, index])
            for index, name in enumerate(('pixel_auroc_hist2048', 'pixel_ap_hist2048', 'image_auroc'))
            for side, bound in enumerate(('low', 'high'))}


def resolve_artifact(class_name, encoder_path):
    checkpoint_path = find_nf_head_checkpoint(class_name, base_dir=str(NF_ROOT))
    if checkpoint_path is None:
        raise FileNotFoundError(f'Head ausente: {class_name} em {NF_ROOT}')
    payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    hp = read_cflow_hparams(payload)
    plan = payload.get('training_recipe') or {}
    recipe = payload.get('inference_recipe') or plan.get('inference')
    if recipe is None:
        if FALLBACK_RECIPE is None:
            raise ValueError(f'{class_name}: checkpoint sem receita. Defina FALLBACK_RECIPE explicitamente.')
        recipe = resolve_training_recipe(class_name, FALLBACK_RECIPE)['inference']
    recipe = dict(recipe)
    trained = hp.get('levels') or [0, 1, 2]
    if any(int(level) not in trained for level in recipe['levels']):
        raise ValueError(f'{class_name}: niveis da receita nao foram treinados')
    if recipe.get('margin', 0) != 0 or recipe.get('pos_calib', 'none') != 'none':
        raise ValueError('Este notebook exige teste completo sem margem e sem calibracao posicional adicional.')
    stats = payload.get('inference_level_stats') if payload.get('inference_recipe') == recipe else None
    return dict(class_name=class_name, checkpoint=str(checkpoint_path), encoder=str(encoder_path),
                payload=payload, hp=hp, recipe=recipe, trained=trained, stats=stats)


encoder_paths = resolve_se_checkpoints(SE_ROOT, CLASSES)
ARTIFACTS = [resolve_artifact(name, encoder_paths[name]) for name in CLASSES]
display(pd.DataFrame([dict(classe=item['class_name'], head=item['checkpoint'], encoder=item['encoder'],
                           epoca=item['payload'].get('epoch'), receita=item['recipe']) for item in ARTIFACTS]))

,classe,head,encoder,epoca,receita
0,glass-insulator,c:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,C:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,10,"{'pad': 112, 'pad_mode': 'reflect', 'tta': 'fl..."
1,lightning-rod-suspension,c:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,C:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,30,"{'pad': 112, 'pad_mode': 'reflect', 'tta': 'fl..."
2,polymer-insulator-upper-shackle,c:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,C:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,27,"{'pad': 112, 'pad_mode': 'reflect', 'tta': 'fl..."
3,vari-grip,c:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,C:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,5,"{'pad': 112, 'pad_mode': 'reflect', 'tta': 'fl..."
4,yoke-suspension,c:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,C:\Users\teo-s\Documents\GitHub\anomaly-v3\ano...,2,"{'pad': 112, 'pad_mode': 'reflect', 'tta': 'fl..."


In [15]:
def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_class_results(name, table, maps, masks, metrics, curve, thresholds, sample, folder):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes[0, 0].plot(curve['fpr'], curve['recall'])
    axes[0, 0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0, 0].set(xlabel='FPR', ylabel='TPR', title=f'{name}: ROC pixel ({PIXEL_BACKEND})')
    axes[0, 1].plot(curve['recall'], curve['precision'])
    axes[0, 1].axhline(metrics['pixel_prevalence'], color='gray', linestyle='--', label='Prevalencia')
    axes[0, 1].set(xlabel='Recall', ylabel='Precisao', title='Precision-recall pixel'); axes[0, 1].legend()
    for label, group in table.groupby('image_label'):
        axes[0, 2].hist(group['image_score'], bins=30, alpha=.6, label='Anomala' if label else 'Normal')
    axes[0, 2].axvline(metrics['image_threshold'], color='black', linestyle='--')
    axes[0, 2].set(title='Score por imagem (nao e probabilidade)', xlabel='Score', ylabel='Imagens'); axes[0, 2].legend()
    finite = np.isfinite(thresholds)
    axes[1, 0].plot(thresholds[finite], curve['f1'][finite])
    axes[1, 0].axvline(metrics['pixel_threshold'], color='black', linestyle='--')
    axes[1, 0].set(title='F1 pixel versus limiar (teste)', xlabel='Limiar', ylabel='F1')
    matrix = np.array([[metrics['pixel_tn'], metrics['pixel_fp']], [metrics['pixel_fn'], metrics['pixel_tp']]])
    image = axes[1, 1].imshow(np.log1p(matrix), cmap='Blues')
    for row in range(2):
        for column in range(2):
            axes[1, 1].text(column, row, f'{matrix[row, column]:,}', ha='center', va='center')
    axes[1, 1].set(xticks=[0, 1], yticks=[0, 1], xlabel='Predito', ylabel='Real', title='Confusao pixel (cor log1p)')
    fig.colorbar(image, ax=axes[1, 1])
    axes[1, 2].hist(table['nll_ms'], bins=30)
    axes[1, 2].set(title='Backbone + NF + TTA, sem I/O/pos-processamento', xlabel='ms por imagem', ylabel='Imagens')
    save_figure(fig, folder / 'curvas_distribuicoes_confusao.png')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    image_curve_precision, image_curve_recall, _ = precision_recall_curve(table['image_label'], table['image_score'])
    from sklearn.metrics import roc_curve
    image_fpr, image_tpr, _ = roc_curve(table['image_label'], table['image_score'])
    axes[0].plot(image_fpr, image_tpr); axes[0].set(title='ROC por imagem (mapa)', xlabel='FPR', ylabel='TPR')
    axes[1].plot(image_curve_recall, image_curve_precision); axes[1].set(title='PR por imagem (mapa)', xlabel='Recall', ylabel='Precisao')
    save_figure(fig, folder / 'curvas_imagem.png')
    fp_cases = table[table.image_label == 0].sort_values('fp', ascending=False).head(2)
    fn_cases = table[table.image_label == 1].sort_values('recall', na_position='last').head(2)
    boundary = table.assign(distance=(table.image_score - metrics['image_threshold']).abs()).sort_values('distance').head(EXAMPLES_PER_CLASS)
    indices = list(dict.fromkeys([*fp_cases.index, *fn_cases.index, *boundary.index]))[:EXAMPLES_PER_CLASS]
    low, high = np.quantile(sample, [.01, .99])
    high = max(high, low + 1e-8)
    if not indices:
        return
    fig, axes = plt.subplots(len(indices), 5, figsize=(17, 3.1 * len(indices)), squeeze=False)
    for row, index in enumerate(indices):
        with Image.open(table.loc[index, 'path']) as original:
            rgb = np.asarray(original.convert('RGB').resize((maps.shape[2], maps.shape[1])))
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(Path(table.loc[index, 'path']).name, fontsize=8)
        axes[row, 1].imshow(masks[index], cmap='gray', vmin=0, vmax=1); axes[row, 1].set_title('Ground truth')
        heat = axes[row, 2].imshow(maps[index], cmap='inferno', vmin=low, vmax=high); axes[row, 2].set_title('Score bruto (q1-q99 da classe)')
        axes[row, 3].imshow(rgb)
        axes[row, 3].imshow(maps[index], cmap='inferno', vmin=low, vmax=high, alpha=.5); axes[row, 3].set_title('Overlay')
        axes[row, 4].imshow(maps[index] >= metrics['pixel_threshold'], cmap='gray', vmin=0, vmax=1)
        axes[row, 4].set_title(f"Predicao | IoU={table.loc[index, 'iou']:.3f}")
        for axis in axes[row]:
            axis.set_axis_off()
    fig.colorbar(heat, ax=axes[:, 2].tolist(), fraction=.02, label='Score NF; nao probabilidade')
    fig.savefig(folder / 'exemplos_heatmaps.png', dpi=150, bbox_inches='tight')
    display(fig); plt.close(fig)
    np.savez_compressed(folder / 'exemplos_mapas.npz', indices=indices, scores=np.asarray(maps[indices]), masks=np.asarray(masks[indices]))
    ranking = table.assign(image_error=table.image_label != table.image_prediction)
    ranking.sort_values(['image_error', 'fp', 'fn'], ascending=False).to_csv(folder / 'ranking_erros.csv', index=False)

In [16]:
def evaluate_artifact(artifact):
    name, recipe, hp = artifact['class_name'], artifact['recipe'], artifact['hp']
    folder = OUTPUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    payload = artifact['payload']
    declared_class = (payload.get('training_recipe') or {}).get('class_name')
    if declared_class is not None and declared_class != name:
        raise ValueError(f'Head da classe {declared_class}, nao {name}')
    size = int(hp.get('img_size') or 448)
    c.img_size = (size, size)
    trainset, testset, labels = build_eval_datasets(name, str(DATASET_ROOT), n_transforms=1, limit=None, seed=SEED)
    trainset.transform = eval_transform()
    trainset.n_transforms = 1
    trainset.get_fixed = False
    paths = [str(record[0]) for record in testset.samples]
    assert len(paths) == len(testset) == len(labels), 'Inventario de teste inconsistente'
    selection_paths = {normalized_path(path) for path in payload.get('selection_validation_paths', [])}
    validation = {index for index, path in enumerate(paths)
                  if normalized_path(Path(path).relative_to(DATASET_ROOT)) in selection_paths}
    model = load_se_model(artifact['encoder'])
    state = payload.get('cflow_state_dict', payload.get('model_state_dict'))
    channels = [[64, 192, 256][level] for level in artifact['trained']]
    backbone = SEBackboneFeatureExtractor(model, out_size=hp.get('out_size') or 96,
                  reflect_pad=recipe['pad'], attention=hp.get('attention') or 'se',
                  pad_mode=recipe['pad_mode'], feat_pool=hp.get('feat_pool') or 0).to(DEVICE).eval()
    head = CFlowPixelHead(channels, cond_dim=hp.get('cond_dim') or 64,
                         n_blocks=hp.get('n_blocks') or infer_n_blocks(state),
                         hidden=hp.get('hidden') or 256, clamp_scale=hp.get('clamp_scale') or .5).to(DEVICE)
    head.load_state_dict(state)
    head.eval()
    stats = artifact['stats']
    if recipe['score_mode'] == 'per_level_std' and stats is None:
        position_stats = compute_positional_stats(backbone, head, eval_loader(trainset), DEVICE,
                          levels=artifact['trained'], feat_stats=payload.get('feat_stats'))
        stats = []
        for mean_map, std_map in position_stats:
            mean = float(mean_map.mean())
            variance = max(float((std_map ** 2 + mean_map ** 2).mean()) - mean ** 2, 0.)
            stats.append((mean, math.sqrt(variance)))
    native_size = int(hp.get('out_size') or 96)
    dtype = np.float64 if recipe['score_mode'] == 'cflow_original' else np.float32
    files = [folder / 'native_nll.npy', folder / 'maps.npy', folder / 'masks.npy']
    native = np.lib.format.open_memmap(files[0], mode='w+', dtype=np.float32,
                                      shape=(len(testset), len(channels), native_size, native_size))
    maps = np.lib.format.open_memmap(files[1], mode='w+', dtype=dtype, shape=(len(testset), size, size))
    masks = np.lib.format.open_memmap(files[2], mode='w+', dtype=np.uint8, shape=maps.shape)
    times = []
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            for index, (images, _, mask) in enumerate(tqdm(eval_loader(testset), desc=f'{name}: teste completo')):
                images = images.to(DEVICE)
                if images.ndim == 5:
                    images = images[:, 0]
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                begin = time.perf_counter()
                nll = nll_with_tta(backbone, head, images, levels=artifact['trained'],
                                  feat_stats=payload.get('feat_stats'), tta=recipe['tta'],
                                  per_channel=recipe['score_mode'] == 'per_level_minmax')
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                times.append((time.perf_counter() - begin) * 1000)
                for level, value in enumerate(nll):
                    if not torch.isfinite(value).all():
                        raise ValueError(f'NLL nao finita: {paths[index]}')
                    native[index, level] = value[0].float().cpu().numpy()
                masks[index] = (mask[0].squeeze().numpy() > .5).astype(np.uint8)
        native.flush()
        selected = [artifact['trained'].index(int(level)) for level in recipe['levels']]
        original = recipe['score_mode'] == 'cflow_original'
        maxima = {level: -float(native[:, level].min()) / channels[level] for level in selected} if original else {}
        global_maximum = -np.inf
        for index in tqdm(range(len(testset)), desc=f'{name}: agregacao'):
            levels_nll = [torch.from_numpy(np.array(native[index:index+1, level], copy=True)) for level in range(len(channels))]
            if original:
                probability = sum(F.interpolate(torch.exp(-levels_nll[level].double() / channels[level] - maxima[level]).unsqueeze(1),
                                                size=(size, size), mode='bilinear', align_corners=recipe['align_corners']).squeeze(1)
                                  for level in selected)
                global_maximum = max(global_maximum, float(probability.max()))
                maps[index] = -probability[0].numpy()
            else:
                result = CFlowPixelHead.aggregate_nll(levels_nll, channels, size, mode=recipe['score_mode'],
                           level_stats=stats, levels=selected, align_corners=recipe['align_corners'])
                maps[index] = gaussian_smooth(result.numpy(), sigma=recipe['sigma'])[0]
        if original:
            for index in range(len(testset)):
                maps[index] = gaussian_smooth((np.asarray(maps[index]) + global_maximum)[None], sigma=recipe['sigma'])[0]
        maps.flush(); masks.flush()
        if not all(np.isfinite(score).all() for score in maps):
            raise ValueError('Mapa final nao finito')
        metrics, curve, thresholds, boot_pos, boot_neg, pixel_auroc_images, sample = pixel_histograms(maps, masks, validation)
        image_scores = np.array([image_score_from_map(np.asarray(score)[None], mode=recipe['image_score'],
                                             top_percent=recipe.get('topk_percent', 1.))[0] for score in maps])
        fit_indices = sorted(validation)
        image_threshold_source = 'validation_reused_in_full_test'
        if not fit_indices or len(np.unique(labels[fit_indices])) < 2:
            fit_indices, image_threshold_source = list(range(len(labels))), 'test_oracle_no_validation_metadata'
        image_threshold = fit_image_threshold(labels[fit_indices], image_scores[fit_indices])
        metrics.update(image_threshold=image_threshold, image_threshold_source=image_threshold_source)
        rows = []
        for index, (score, mask) in enumerate(zip(maps, masks)):
            predicted, actual = score >= metrics['pixel_threshold'], mask.astype(bool)
            counts = confusion_metrics(np.sum(predicted & actual), np.sum(predicted & ~actual),
                                       np.sum(~predicted & actual), np.sum(~predicted & ~actual))
            rows.append(dict(index=index, path=paths[index], defect=Path(paths[index]).parent.name,
                             image_label=int(labels[index]), image_score=float(image_scores[index]),
                             image_prediction=int(image_scores[index] >= image_threshold),
                             used_for_epoch_selection=index in validation, pixel_auroc=pixel_auroc_images[index],
                             nll_ms=times[index], **counts))
        table = pd.DataFrame(rows).set_index('index')
        totals = table[['tp', 'fp', 'fn', 'tn']].sum()
        metrics.update({f'pixel_{key}': value for key, value in confusion_metrics(**totals.to_dict()).items()})
        metrics.update({f'image_{key}': value for key, value in binary_image_metrics(labels, image_scores, image_threshold).items()})
        metrics.update(image_bootstrap(labels, image_scores, boot_pos, boot_neg))
        metrics['aupro_cflow_global'] = compute_aupro_cflow(maps, masks) if COMPUTE_AUPRO else np.nan
        metrics['aupro_rd_legacy'] = (float(np.mean([compute_pro_rd_style(mask, score) for mask, score in zip(masks, maps) if mask.any()]))
                                             if COMPUTE_LEGACY_AUPRO else np.nan)
        metrics.update(class_name=name, n_images=len(testset), n_anomaly=int(labels.sum()), n_validation_reused=len(validation),
                       checkpoint_epoch=payload.get('epoch'), nll_ms_mean=float(np.mean(times)),
                       nll_ms_p95=float(np.percentile(times, 95)), evaluation_seconds=time.perf_counter() - started,
                       recipe=json.dumps(recipe), bootstrap_method='stratified_images_hist2048_not_training_seeds')
        metrics['pixel_ap_lift_over_prevalence'] = safe_ratio(metrics['pixel_ap'], metrics['pixel_prevalence'])
        table.to_csv(folder / 'per_image.csv')
        pd.DataFrame(dict(fpr=curve['fpr'], recall=curve['recall'], precision=curve['precision'],
                          f1=curve['f1'], threshold=thresholds)).to_csv(folder / 'pixel_curves.csv', index=False)
        metadata = dict(dataset=str(DATASET_ROOT), head=artifact['checkpoint'], encoder=artifact['encoder'],
                        head_sha256=sha256_file(artifact['checkpoint']), encoder_sha256=sha256_file(artifact['encoder']),
                        recipe=recipe, level_stats=stats, metric_backend=PIXEL_BACKEND,
                        n_bins=N_BINS, bootstrap=BOOTSTRAP, stride=1, margin=0,
                        cohort='complete_test', warning='descriptive evaluation; validation reused; no independent holdout')
        (folder / 'evaluation_config.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        plot_class_results(name, table, maps, masks, metrics, curve, thresholds, sample, folder)
        return metrics, table, curve
    finally:
        for array in (native, maps, masks):
            array.flush()
            array._mmap.close()
        if not KEEP_MAPS:
            for path in files:
                path.unlink(missing_ok=True)
        del head, backbone, model
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

## Executar a avaliacao completa

A proxima celula executa inferencia e todas as metricas habilitadas para **cada imagem e cada pixel do teste**, uma classe por vez. Pode demorar. AUPRO global nao e o mesmo AUPRO per-image dos scripts antigos. Mapas em cache sao apagados ao final de cada classe se `KEEP_MAPS=False`; os exemplos, as tabelas e as curvas permanecem.

Limiares sao escolhidos nas imagens de validacao registradas no checkpoint, quando disponiveis. Como estas imagens fazem parte do teste completo, os resultados continuam descritivos. Sem esses caminhos, o notebook declara explicitamente um limiar-oraculo do teste. Os ICs reamostram imagens (bins reduzidos para 2048 nos ICs de pixel); nao medem variacao de treino, de seeds ou de escolha da receita, e nao ha IC de AUPRO.

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS, PER_IMAGE, CURVES = [], [], {}
for artifact in ARTIFACTS:
    print('\n' + '=' * 80, '\n', artifact['class_name'], artifact['recipe'])
    metrics, per_image, curve = evaluate_artifact(artifact)
    RESULTS.append(metrics)
    PER_IMAGE.append(per_image.assign(class_name=artifact['class_name']))
    CURVES[artifact['class_name']] = curve
    pd.DataFrame(RESULTS).to_csv(OUTPUT_DIR / 'results_per_class.csv', index=False)
    pd.concat(PER_IMAGE).to_csv(OUTPUT_DIR / 'per_image_all.csv', index=False)
    display(pd.DataFrame([metrics]).drop(columns='recipe').T.rename(columns={0: 'valor'}))
print('Avaliacao concluida:', OUTPUT_DIR)


 glass-insulator {'pad': 112, 'pad_mode': 'reflect', 'tta': 'flips', 'score_mode': 'per_level_std', 'levels': '02', 'align_corners': True, 'sigma': 8.0, 'image_score': 'max', 'topk_percent': 1.0, 'margin': 0, 'pos_calib': 'none'}


c:\Users\teo-s\.conda\envs\anomaly_attentideffernet\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\teo-s\.conda\envs\anomaly_attentideffernet\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Histogramas completos:  89%|████████▉ | 576/649 [00:08<00:01, 69.10it/s]C:\Users\teo-s\AppData\Local\Temp\ipykernel_31808\3904335704.py:82: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auroc=float(np.trapz(recall, fpr)) if pos

<Figure size 1600x900 with 7 Axes>

<Figure size 1200x400 with 2 Axes>

<Figure size 1700x1860 with 31 Axes>

...

<Figure size 1600x900 with 7 Axes>

<Figure size 1200x400 with 2 Axes>

<Figure size 1700x1860 with 31 Axes>

,valor
pixel_auroc,0.925244
pixel_ap,0.019691
pixel_prevalence,0.000746
pixel_best_f1_test_oracle,0.060523
pixel_threshold_test_oracle,8.67917
...,...
nll_ms_mean,173.010641
nll_ms_p95,183.51714
evaluation_seconds,259.218739
bootstrap_method,stratified_images_hist2048_not_training_seeds


Avaliacao concluida: c:\Users\teo-s\Documents\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\resultado_analise_final\pixel_notebook\run_20261026_180544


In [18]:
results = pd.DataFrame(RESULTS).set_index('class_name')
images = pd.concat(PER_IMAGE).reset_index()
metric_columns = ['pixel_auroc', 'pixel_ap', 'pixel_prevalence', 'pixel_ap_lift_over_prevalence',
                  'aupro_cflow_global', 'pixel_accuracy', 'pixel_precision', 'pixel_recall',
                  'pixel_f1', 'pixel_f1_macro_binary', 'pixel_f1_micro_binary', 'pixel_f1_weighted',
                  'pixel_iou', 'pixel_miou', 'pixel_dice', 'image_auroc', 'image_ap',
                  'image_accuracy', 'image_precision', 'image_recall', 'image_f1', 'nll_ms_mean']
summary = results[metric_columns].copy()
summary.loc['MACRO_CLASSES'] = summary.mean(axis=0, numeric_only=True)
display(summary.style.format('{:.4f}', na_rep='nao calculado').background_gradient(cmap='YlGnBu', axis=0))
summary.to_csv(OUTPUT_DIR / 'summary_macro.csv')
ci_columns = [column for column in results if '_ci_' in column]
display(results[['n_images', 'n_anomaly', 'n_validation_reused', 'threshold_source',
                 'pixel_ranking_backend', *ci_columns]])

pooled_counts = images[['tp', 'fp', 'fn', 'tn']].sum()
pooled = confusion_metrics(**pooled_counts.to_dict())
micro = pd.DataFrame([dict(scope='PIXELS_MICRO_ALL_CLASSES', **pooled)])
display(micro)
micro.to_csv(OUTPUT_DIR / 'micro_confusion_metrics.csv', index=False)
print('AUROC/AP nao sao agrupadas entre classes: os scores tem escalas diferentes. MACRO_CLASSES e media por classe.')

subgroups = []
for (class_name, defect), group in images.groupby(['class_name', 'defect']):
    counts = group[['tp', 'fp', 'fn', 'tn']].sum()
    subgroups.append(dict(class_name=class_name, defect=defect, n_images=len(group),
                          n_anomaly=int(group.image_label.sum()), **confusion_metrics(**counts.to_dict())))
subgroup_table = pd.DataFrame(subgroups)
display(subgroup_table)
subgroup_table.to_csv(OUTPUT_DIR / 'metrics_by_defect.csv', index=False)

images['score_quartile'] = images.groupby('class_name')['image_score'].transform(
    lambda series: pd.qcut(series.rank(method='first'), min(4, len(series)), labels=False))
score_groups = images.groupby(['class_name', 'score_quartile']).agg(
    n_images=('image_label', 'size'), anomaly_rate=('image_label', 'mean'),
    mean_score=('image_score', 'mean'), mean_latency_ms=('nll_ms', 'mean'))
display(score_groups)
score_groups.to_csv(OUTPUT_DIR / 'metrics_by_score_quartile.csv')
print('Quartis sao faixas de score, nao faixas de probabilidade/confianca calibrada.')

,pixel_auroc,pixel_ap,pixel_prevalence,pixel_ap_lift_over_prevalence,aupro_cflow_global,pixel_accuracy,pixel_precision,pixel_recall,pixel_f1,pixel_f1_macro_binary,pixel_f1_micro_binary,pixel_f1_weighted,pixel_iou,pixel_miou,pixel_dice,image_auroc,image_ap,image_accuracy,image_precision,image_recall,image_f1,nll_ms_mean
class_name,,,,,,,,,,,,,,,,,,,,,,
glass-insulator,0.6800,0.0013,0.0002,7.3816,0.5999,0.9793,0.0025,0.2893,0.0050,0.4973,0.9793,0.9894,0.0025,0.4909,0.0050,0.7501,0.2995,0.8783,0.4000,0.3939,0.3969,207.2462
lightning-rod-suspension,0.9208,0.0416,0.0057,7.2717,0.7268,0.9726,0.0511,0.2160,0.0826,0.5343,0.9726,0.9809,0.0431,0.5078,0.0826,0.7527,0.5225,0.7259,0.5263,0.5128,0.5195,195.8888
polymer-insulator-upper-shackle,0.6671,0.0146,0.0067,2.1645,0.4168,0.9407,0.0241,0.1973,0.0429,0.5061,0.9407,0.9632,0.0219,0.4813,0.0429,0.6004,0.3024,0.4770,0.2798,0.8571,0.4219,197.5923
vari-grip,0.8972,0.0759,0.0121,6.2902,0.7329,0.8820,0.0676,0.6866,0.1231,0.5299,0.8820,0.9269,0.0656,0.4733,0.1231,0.5947,0.4287,0.2597,0.2597,1.0000,0.4124,197.3051
yoke-suspension,0.9252,0.0197,0.0007,26.3963,0.8262,0.9974,0.0400,0.1052,0.0579,0.5283,0.9974,0.9980,0.0298,0.5136,0.0579,0.7842,0.1237,0.9458,0.2162,0.1818,0.1975,173.0106
MACRO_CLASSES,0.8181,0.0306,0.0051,9.9008,0.6605,0.9544,0.0370,0.2989,0.0623,0.5192,0.9544,0.9717,0.0326,0.4934,0.0623,0.6964,0.3354,0.6574,0.3364,0.5891,0.3896,194.2086


,n_images,n_anomaly,n_validation_reused,threshold_source,pixel_ranking_backend,pixel_auroc_hist2048_ci_low,pixel_auroc_hist2048_ci_high,pixel_ap_hist2048_ci_low,pixel_ap_hist2048_ci_high,image_auroc_ci_low,image_auroc_ci_high
class_name,,,,,,,,,,,
glass-insulator,649,66,259,validation_reused_in_full_test,histogram,0.552409,0.802348,0.000662,0.002336,0.699692,0.820879
lightning-rod-suspension,135,39,54,validation_reused_in_full_test,histogram,0.908645,0.933199,0.031197,0.062655,0.645373,0.833901
polymer-insulator-upper-shackle,283,63,113,validation_reused_in_full_test,histogram,0.616973,0.715875,0.008934,0.021405,0.522624,0.688075
vari-grip,154,40,62,validation_reused_in_full_test,histogram,0.875166,0.915514,0.063404,0.102485,0.477072,0.701107
yoke-suspension,1199,44,480,validation_reused_in_full_test,histogram,0.850529,0.963310,0.013340,0.030785,0.711402,0.856638


,scope,tp,fp,fn,tn,accuracy,precision,recall,specificity,balanced_accuracy,f1,dice,f1_normal,f1_macro_binary,f1_weighted,f1_micro_binary,iou,iou_normal,miou,fpr
0,PIXELS_MICRO_ALL_CLASSES,390428,10345925,722369,474244958,0.977212,0.036365,0.350853,0.97865,0.664752,0.0659,0.0659,0.988465,0.527182,0.986352,0.977212,0.034073,0.977194,0.505633,0.02135


AUROC/AP nao sao agrupadas entre classes: os scores tem escalas diferentes. MACRO_CLASSES e media por classe.


,class_name,defect,n_images,n_anomaly,tp,fp,fn,tn,accuracy,precision,recall,specificity,balanced_accuracy,f1,dice,f1_normal,f1_macro_binary,f1_weighted,f1_micro_binary,iou,iou_normal,miou,fpr
0,glass-insulator,good,583,0,0,1770219,0,115240213,0.984871,0.000000,NaN,0.984871,NaN,0.000000,0.000000,0.992378,0.496189,0.992378,0.984871,0.000000,0.984871,0.492436,0.015129
1,glass-insulator,missing-cap,66,66,6716,909095,16502,12314151,0.930125,0.007333,0.289258,0.931250,0.610254,0.014304,0.014304,0.963779,0.489041,0.962114,0.930125,0.007204,0.930090,0.468647,0.068750
2,lightning-rod-suspension,good,96,0,0,317645,0,18949939,0.983514,0.000000,NaN,0.983514,NaN,0.000000,0.000000,0.991688,0.495844,0.991688,0.983514,0.000000,0.983514,0.491757,0.016486
3,lightning-rod-suspension,rust,39,39,33463,304040,121494,7368459,0.945636,0.099149,0.215950,0.960373,0.588161,0.135901,0.135901,0.971935,0.553918,0.955384,0.945636,0.072905,0.945402,0.509153,0.039627
4,polymer-insulator-upper-shackle,good,220,0,0,2233894,0,41920986,0.949408,0.000000,NaN,0.949408,NaN,0.000000,0.000000,0.974047,0.487024,0.974047,0.949408,0.000000,0.949408,0.474704,0.050592
5,polymer-insulator-upper-shackle,rust,63,63,75450,827832,306938,11434132,0.910255,0.083529,0.197313,0.932488,0.564900,0.117371,0.117371,0.952724,0.535047,0.927461,0.910255,0.062344,0.909716,0.486030,0.067512
6,vari-grip,good,114,0,0,2693036,0,20187220,0.882299,0.000000,NaN,0.882299,NaN,0.000000,0.000000,0.937469,0.468735,0.937469,0.882299,0.000000,0.882299,0.441149,0.117701
7,vari-grip,rust,40,40,255914,836682,116802,6818762,0.881233,0.234226,0.686619,0.890708,0.788663,0.349296,0.349296,0.934653,0.641974,0.907477,0.881233,0.211604,0.877322,0.544463,0.109292
8,yoke-suspension,good,1155,0,0,393210,0,231419910,0.998304,0.000000,NaN,0.998304,NaN,0.000000,0.000000,0.999151,0.499576,0.999151,0.998304,0.000000,0.998304,0.499152,0.001696
9,yoke-suspension,rust,44,44,18885,60272,160633,8591186,0.974985,0.238576,0.105198,0.993033,0.549116,0.146013,0.146013,0.987307,0.566660,0.970205,0.974985,0.078756,0.974932,0.526844,0.006967


n_images  anomaly_rate  \
class_name                      score_quartile                           
glass-insulator                 0                    163      0.030675   
                                1                    162      0.074074   
                                2                    162      0.055556   
                                3                    162      0.246914   
lightning-rod-suspension        0                     34      0.088235   
                                1                     34      0.176471   
                                2                     33      0.333333   
                                3                     34      0.558824   
polymer-insulator-upper-shackle 0                     71      0.098592   
                                1                     71      0.267606   
                                2                     70      0.285714   
                                3                     71      0.239437   
vari-grip                       0                     39      0.230769   
                                1                     38      0.184211   
                                2                     38      0.236842   
                                3                     39      0.384615   
yoke-suspension                 0                    300      0.006667   
                                1                    300      0.013333   
                                2                    299      0.023411   
                                3                    300      0.103333   

                                                mean_score  mean_latency_ms  
class_name                      score_quartile                               
glass-insulator                 0                 3.721360       206.950894  
                                1                 4.830222       207.549355  
                                2                 5.897606       207.065238  
                                3                 8.283629       207.421075  
lightning-rod-suspension        0                 6.428739       189.551594  
                                1                 8.159711       195.881026  
                                2                 9.799675       199.129664  
                                3                12.986256       199.088224  
polymer-insulator-upper-shackle 0                 5.286137       195.322454  
                                1                 5.693420       198.439482  
                                2                 7.492355       199.086116  
                                3                12.293126       197.542145  
vari-grip                       0                 2.389979       195.947667  
                                1                 2.741308       196.985145  
                                2                 3.024583       197.794171  
                                3                 3.895908       198.497738  
yoke-suspension                 0                 6.033962       172.323563  
                                1                 7.490671       172.366770  
                                2                 8.815713       172.998220  
                                3                11.269652       174.353971

Quartis sao faixas de score, nao faixas de probabilidade/confianca calibrada.


In [19]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
results[['pixel_auroc', 'pixel_ap', 'aupro_cflow_global']].plot.bar(ax=axes[0, 0])
axes[0, 0].set(title='Metricas de ranking e regiao', ylabel='Score', ylim=(0, 1))
results[['pixel_precision', 'pixel_recall', 'pixel_f1', 'pixel_iou']].plot.bar(ax=axes[0, 1])
axes[0, 1].set(title='Metricas no limiar declarado por classe', ylabel='Score', ylim=(0, 1))
for name, curve in CURVES.items():
    axes[1, 0].plot(curve['fpr'], curve['recall'], label=name)
    axes[1, 1].plot(curve['recall'], curve['precision'], label=name)
axes[1, 0].set(title='ROC pixel por classe', xlabel='FPR', ylabel='TPR')
axes[1, 1].set(title='Precision-recall pixel por classe', xlabel='Recall', ylabel='Precisao')
axes[1, 0].legend(fontsize=8); axes[1, 1].legend(fontsize=8)
save_figure(fig, OUTPUT_DIR / 'comparativo_classes.png')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
results[['nll_ms_mean', 'nll_ms_p95']].plot.bar(ax=axes[0])
axes[0].set(title='Latencia observada de backbone/NF/TTA', ylabel='ms/imagem')
errors = images.assign(error_fraction=(images.fp + images.fn) / (images.tp + images.fp + images.fn + images.tn))
for name, group in errors.groupby('class_name'):
    axes[1].hist(group.error_fraction, bins=25, alpha=.5, label=name)
axes[1].set(title='Fracao de pixels errados por imagem', xlabel='(FP+FN)/pixels', ylabel='Imagens')
axes[1].legend(fontsize=8)
save_figure(fig, OUTPUT_DIR / 'latencia_erros.png')

false_positives = images[(images.image_label == 0) & (images.image_prediction == 1)].sort_values('fp', ascending=False)
false_negatives = images[(images.image_label == 1) & (images.image_prediction == 0)].sort_values('fn', ascending=False)
display(Markdown('### Falsos positivos por imagem'))
display(false_positives[['class_name', 'path', 'image_score', 'fp', 'pixel_auroc']].head(20))
display(Markdown('### Falsos negativos por imagem'))
display(false_negatives[['class_name', 'path', 'image_score', 'fn', 'recall', 'iou']].head(20))
false_positives.to_csv(OUTPUT_DIR / 'false_positives.csv', index=False)
false_negatives.to_csv(OUTPUT_DIR / 'false_negatives.csv', index=False)

<Figure size 1500x1000 with 4 Axes>

<Figure size 1300x400 with 2 Axes>

### Falsos positivos por imagem

,class_name,path,image_score,fp,pixel_auroc
790,polymer-insulator-upper-shackle,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,23.447184,47314,NaN
862,polymer-insulator-upper-shackle,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,10.472951,46261,NaN
857,polymer-insulator-upper-shackle,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,7.031842,40796,NaN
925,polymer-insulator-upper-shackle,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,27.881741,38070,NaN
1107,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,3.565028,37600,NaN
1114,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,3.293361,36727,NaN
1141,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,2.836469,35820,NaN
1111,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,3.456540,35355,NaN
1134,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,3.073817,35235,NaN
1093,vari-grip,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,3.731222,34517,NaN


### Falsos negativos por imagem

,class_name,path,image_score,fn,recall,iou
2398,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,7.886881,7197,0.000000,0.000000
2399,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,9.772624,6721,0.037795,0.036254
2395,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,8.190796,6504,0.000000,0.000000
2406,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,12.579770,5981,0.230937,0.145673
771,lightning-rod-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,8.627739,5939,0.091757,0.062722
1057,polymer-insulator-upper-shackle,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,5.282779,5900,0.076249,0.028362
2400,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,10.968184,5692,0.110486,0.094633
2401,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,9.730673,5667,0.017340,0.016966
2397,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,6.905448,5615,0.000000,0.000000
2407,yoke-suspension,C:\Users\teo-s\Documents\GitHub\anomaly-detect...,9.795908,5586,0.023427,0.020024


In [20]:
def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


report = dict(created_at=datetime.now().isoformat(), python=sys.version, platform=platform.platform(),
              torch_version=torch.__version__, device=DEVICE, dataset=str(DATASET_ROOT),
              nf_root=str(NF_ROOT), classes=CLASSES, scope='complete_test_descriptive_not_independent_holdout',
              pixel_backend=PIXEL_BACKEND, pixel_bins=N_BINS, bootstrap_images=BOOTSTRAP,
              warnings=['AUROC/AP por histograma sao aproximadas no modo histogram.',
                        'ICs de pixel usam bins reduzidos e condicionam a receita/coorte fixa.',
                        'AUPRO global usa thresholds lineares; zeros exigem auditoria, nao correcao silenciosa.',
                        'Melhor F1 no teste e diagnostico-oraculo; limiares declarados podem reutilizar validacao.',
                        'Latencia nao tem warmup padronizado e exclui I/O/pos-processamento.'],
              results=RESULTS)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(json_ready(report), indent=2, allow_nan=False), encoding='utf-8')
(OUTPUT_DIR / 'summary.html').write_text(summary.to_html(float_format=lambda value: f'{value:.4f}'), encoding='utf-8')
try:
    with pd.ExcelWriter(OUTPUT_DIR / 'evaluation_tables.xlsx', engine='openpyxl') as writer:
        results.to_excel(writer, sheet_name='all_metrics')
        summary.to_excel(writer, sheet_name='summary_macro')
        images.to_excel(writer, sheet_name='per_image', index=False)
        subgroup_table.to_excel(writer, sheet_name='by_defect', index=False)
        micro.to_excel(writer, sheet_name='micro', index=False)
except ImportError:
    print('Excel opcional indisponivel: instale openpyxl no kernel. CSV/HTML/JSON foram preservados.')
from html import escape
parts = ['<html><meta charset="utf-8"><body><h1>Avaliacao das pixel heads</h1>',
         '<p>Teste completo, avaliacao descritiva. Nao e holdout independente. Scores nao sao probabilidades.</p>',
         summary.to_html(float_format=lambda value: f'{value:.4f}'),
         '<img src="comparativo_classes.png" width="100%">']
for name in CLASSES:
    parts.extend([f'<h2>{escape(name)}</h2>',
                  f'<img src="{escape(name)}/curvas_distribuicoes_confusao.png" width="100%">',
                  f'<img src="{escape(name)}/exemplos_heatmaps.png" width="100%">'])
parts.append('</body></html>')
(OUTPUT_DIR / 'report.html').write_text('\n'.join(parts), encoding='utf-8')
print('Relatorio:', OUTPUT_DIR / 'report.html')
display(pd.DataFrame([dict(arquivo=str(path.relative_to(OUTPUT_DIR)), tamanho_kb=round(path.stat().st_size / 1024, 1))
                      for path in OUTPUT_DIR.rglob('*') if path.is_file()]))

Relatorio: c:\Users\teo-s\Documents\GitHub\anomaly-v3\anomaly-detection-pesquisa\differnet\resultado_analise_final\pixel_notebook\run_20261026_180544\report.html


,arquivo,tamanho_kb
0,comparativo_classes.png,178.6
1,evaluation_tables.xlsx,436.1
2,false_negatives.csv,57.1
3,false_positives.csv,151.0
4,latencia_erros.png,69.3
5,metrics.json,17.5
6,metrics_by_defect.csv,3.1
7,metrics_by_score_quartile.csv,1.7
8,micro_confusion_metrics.csv,0.5
9,per_image_all.csv,937.5
